# DeepSORT Extended — Google Colab

**Runtime:** GPU (T4). **Данные:** `MyDrive/Videos-CV/`

In [ ]:
# 1. Проверка GPU
!nvidia-smi

In [ ]:
# 2. Клонирование репозитория и установка зависимостей
REPO_URL = "https://github.com/danvlak-batya/deep-sort-project.git"
PROJECT_DIR = "/content/deep-sort-project"

import os
os.chdir("/content")

if not os.path.exists(os.path.join(PROJECT_DIR, "run_tracker.py")):
    if os.path.exists(PROJECT_DIR):
        !rm -rf {PROJECT_DIR}
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print("Рабочая папка:", os.getcwd())
assert os.path.exists("run_tracker.py"), "run_tracker.py не найден — проверьте клонирование"

!pip install -q -r requirements.txt
print("Установка завершена")

In [ ]:
# 3. Подключение Google Drive
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = "/content/drive/MyDrive/Videos-CV"

import os
assert os.path.isdir(DATA_ROOT), f"Папка не найдена: {DATA_ROOT}"
print("Доступные последовательности:")
for name in sorted(os.listdir(DATA_ROOT)):
    path = os.path.join(DATA_ROOT, name)
    if os.path.isdir(path):
        has_img = any(os.path.isdir(os.path.join(path, d)) for d in ("img", "img1", "images"))
        print(f"  - {name}" + ("  ok" if has_img else "  (нет папки img)"))

In [ ]:
# 4. Настройки
import os
from utils.mot_paths import find_sequence_dir, list_image_filenames

SEQUENCE_FOLDER = "Kitti-17"   # имя папки на Drive
SEQUENCE = "KITTI-17"          # имя для файлов результатов

DETECTOR = "yolov8n"
REID = "osnet_x0_25"
CONFIG = "configs/default.yaml"

SEQ_DIR = find_sequence_dir(DATA_ROOT, SEQUENCE_FOLDER)
print("Папка:", SEQ_DIR)
print("Кадров:", len(list_image_filenames(SEQ_DIR)))

In [ ]:
# 5. Трекинг
import os
os.makedirs("results/demo", exist_ok=True)

!python run_tracker.py \
  --sequence_dir "{SEQ_DIR}" \
  --config {CONFIG} \
  --detector {DETECTOR} \
  --reid {REID} \
  --output_file results/demo/{SEQUENCE}.txt

result_path = f"results/demo/{SEQUENCE}.txt"
print("Результат создан:", os.path.exists(result_path))
if os.path.exists(result_path):
    print("Размер файла:", os.path.getsize(result_path), "байт")

In [ ]:
# 6. Overlay-видео
!python tools/generate_overlays.py \
  --mot_dir "{DATA_ROOT}" \
  --results_dir results/demo \
  --output_dir results/overlays \
  --sequence {SEQUENCE}

from IPython.display import Video, display
overlay_path = f"results/overlays/{SEQUENCE}_overlay.mp4"
if os.path.exists(overlay_path):
    display(Video(overlay_path, embed=True))
else:
    print("Видео не найдено:", overlay_path)